In [5]:
!pip install -q --upgrade langchain langchain-groq pandas openpyxl

In [6]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("AgenticAI")

print("Groq API key configured successfully!")

Groq API key configured successfully!


In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("SportsMate AI LLM initialized successfully!")

SportsMate AI LLM initialized successfully!


In [8]:
SPORTS_PRODUCTS = {

    "Cricket": [
        "Cricket Bat", "English Willow Bat", "Kashmir Willow Bat",
        "Leather Cricket Ball", "Tennis Cricket Ball", "Cricket Stumps",
        "Batting Gloves", "Wicket Keeping Gloves", "Cricket Pads",
        "Cricket Helmet", "Thigh Guard", "Cricket Kit Bag",
        "Cricket Shoes", "Arm Guard", "Abdominal Guard"
    ],

    "Football": [
        "Football", "Training Football", "Professional Football",
        "Football Shoes", "Football Shin Guards", "Goalkeeper Gloves",
        "Football Socks", "Football Net", "Football Pump",
        "Football Kit"
    ],

    "Badminton": [
        "Badminton Racket", "Professional Badminton Racket",
        "Badminton Shuttlecock", "Feather Shuttlecock",
        "Nylon Shuttlecock", "Badminton Net", "Racket Grip",
        "Badminton Shoes", "Badminton Kit Bag", "Racket Strings"
    ],

    "Tennis": [
        "Tennis Racket", "Professional Tennis Racket",
        "Tennis Ball", "Tennis Net", "Tennis Shoes",
        "Tennis Grip", "Tennis Strings", "Tennis Kit Bag",
        "Tennis Wristband", "Tennis Headband"
    ],

    "Basketball": [
        "Basketball", "Professional Basketball",
        "Basketball Shoes", "Basketball Hoop",
        "Basketball Net", "Basketball Pump",
        "Basketball Socks", "Basketball Jersey",
        "Basketball Knee Pads", "Basketball Arm Sleeve"
    ],

    "Volleyball": [
        "Volleyball", "Professional Volleyball",
        "Volleyball Net", "Volleyball Shoes",
        "Volleyball Knee Pads", "Volleyball Ankle Support",
        "Volleyball Socks", "Volleyball Arm Sleeves",
        "Volleyball Pump", "Volleyball Kit"
    ],

    "Table Tennis": [
        "Table Tennis Bat", "Professional Table Tennis Bat",
        "Table Tennis Balls", "Table Tennis Net",
        "Table Tennis Table", "Table Tennis Shoes",
        "Table Tennis Cover", "Table Tennis Rubber",
        "Table Tennis Grip", "Table Tennis Kit"
    ],

    "Fitness": [
        "Skipping Rope", "Yoga Mat", "Resistance Band",
        "Exercise Mat", "Push Up Bar", "Ab Roller",
        "Gym Gloves", "Fitness Belt", "Foam Roller",
        "Exercise Ball"
    ],

    "Gym Equipment": [
        "Dumbbells", "Adjustable Dumbbells", "Barbell",
        "Weight Plates", "Kettlebell", "Weight Lifting Belt",
        "Gym Bench", "Pull Up Bar", "Hand Gripper",
        "Wrist Wraps"
    ],

    "Athletics": [
        "Running Shoes", "Athletic Spikes", "Running Shorts",
        "Sports T-Shirt", "Training Cones", "Agility Ladder",
        "Starting Blocks", "Javelin", "Discus",
        "Sports Water Bottle"
    ]
}


# Sample prices for each category

CATEGORY_BASE_PRICES = {
    "Cricket": 149,
    "Football": 199,
    "Badminton": 149,
    "Tennis": 199,
    "Basketball": 299,
    "Volleyball": 199,
    "Table Tennis": 99,
    "Fitness": 149,
    "Gym Equipment": 299,
    "Athletics": 199
}


# Available sample colors

CATEGORY_COLORS = {
    "Cricket": ["Black", "Blue", "Red"],
    "Football": ["Black", "White", "Blue"],
    "Badminton": ["Black", "Red", "White"],
    "Tennis": ["Yellow", "Black", "Blue"],
    "Basketball": ["Orange", "Black"],
    "Volleyball": ["White", "Blue", "Yellow"],
    "Table Tennis": ["Red", "Black"],
    "Fitness": ["Black", "Pink", "Blue"],
    "Gym Equipment": ["Black", "Grey", "Red"],
    "Athletics": ["Black", "White", "Blue"]
}


# Create the product catalog

SPORTS_CATALOG = {}

for category, products in SPORTS_PRODUCTS.items():

    base_price = CATEGORY_BASE_PRICES[category]

    for index, product in enumerate(products):

        price = base_price + (index * 150)

        SPORTS_CATALOG[product] = {
            "category": category,
            "price": price,
            "colors": CATEGORY_COLORS[category],
            "details": f"{product} for {category} activities",
            "availability": "Available"
        }


print("Sports Catalog Created Successfully!")
print("Total Categories:", len(SPORTS_PRODUCTS))
print("Total Products:", len(SPORTS_CATALOG))

Sports Catalog Created Successfully!
Total Categories: 10
Total Products: 105


In [9]:
import pandas as pd

catalog_df = pd.DataFrame([
    {
        "Product": product,
        "Category": details["category"],
        "Price": f"Rs {details['price']}",
        "Colors": ", ".join(details["colors"]),
        "Availability": details["availability"]
    }
    for product, details in SPORTS_CATALOG.items()
])

display(catalog_df)

print("Total products:", len(catalog_df))

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


,Product,Category,Price,Colors,Availability
0,Cricket Bat,Cricket,Rs 149,"Black, Blue, Red",Available
1,English Willow Bat,Cricket,Rs 299,"Black, Blue, Red",Available
2,Kashmir Willow Bat,Cricket,Rs 449,"Black, Blue, Red",Available
3,Leather Cricket Ball,Cricket,Rs 599,"Black, Blue, Red",Available
4,Tennis Cricket Ball,Cricket,Rs 749,"Black, Blue, Red",Available
...,...,...,...,...,...
100,Agility Ladder,Athletics,Rs 949,"Black, White, Blue",Available
101,Starting Blocks,Athletics,Rs 1099,"Black, White, Blue",Available
102,Javelin,Athletics,Rs 1249,"Black, White, Blue",Available
103,Discus,Athletics,Rs 1399,"Black, White, Blue",Available


Total products: 105


In [14]:

import re
import json

def get_relevant_catalog(question, chat_history):

    # Include recent customer questions for follow-up context
    recent_questions = [
        msg.content
        for msg in chat_history
        if isinstance(msg, HumanMessage)
    ][-4:]

    search_text = " ".join(recent_questions + [question]).lower()

    # Check whether the customer has specified a budget
    budget_match = re.search(
        r"(?:under|below|less than|within)\s*(?:rs\.?|₹)?\s*(\d+)",
        search_text
    )

    if budget_match:

        budget = int(budget_match.group(1))

        relevant = {
            name: details
            for name, details in SPORTS_CATALOG.items()
            if details["price"] <= budget
        }

    else:

        stopwords = {
            "how", "much", "is", "the", "a", "an", "do",
            "you", "have", "what", "are", "available",
            "which", "colors", "color", "price", "tell",
            "me", "about", "my", "name", "please", "can",
            "i", "want", "need", "show", "some", "other",
            "details", "product", "products"
        }

        words = set(re.findall(r"[a-z]+", search_text))
        keywords = words - stopwords

        scored_products = []

        for name, details in SPORTS_CATALOG.items():

            product_words = set(
                re.findall(
                    r"[a-z]+",
                    name.lower() + " " + details["category"].lower()
                )
            )

            score = len(keywords.intersection(product_words))

            if score > 0:
                scored_products.append((score, name, details))

        scored_products.sort(
            key=lambda item: item[0],
            reverse=True
        )

        relevant = {
            name: details
            for score, name, details in scored_products[:15]
        }

    return json.dumps(relevant, indent=2)


# Rebuild conversation chain

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()


# Updated ask function

def ask(question: str) -> str:

    relevant_catalog = get_relevant_catalog(
        question,
        history
    )

    answer = chain.invoke({
        "catalog": relevant_catalog,
        "history": history[-6:],
        "question": question
    })

    history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer)
    ])

    return answer


print("Optimized SportsMate AI Conversation Chain Ready!")


Optimized SportsMate AI Conversation Chain Ready!


In [11]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:

    answer = chain.invoke({
        "catalog": catalog_text,
        "history": history,
        "question": question
    })

    history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer)
    ])

    return answer


print("Conversation Chain Created Successfully!")

Conversation Chain Created Successfully!


In [16]:

history.clear()

print(ask("Hi, I'm Jahnavi, I need cricket equipment."))

print(ask("How much is a cricket bat?"))

print(ask("What colors are available?"))

print(ask("Which products are available under Rs 500?"))

print(ask("What is my name?"))


Hi Jahnavi! 👋 I’d love to help you gear up for cricket. Are you looking for bats, gloves, pads, or something else? Let me know any specific items, preferred colors, or budget you have in mind, and I’ll suggest the best options from our catalog.
The regular **Cricket Bat** is priced at **₹149**. It’s available in three colors:

- Black  
- Blue  
- Red  

If you’re interested in premium options, we also have the **English Willow Bat** (₹299) and the **Kashmir Willow Bat** (₹449), each in the same color choices. Let me know which one you’d like or if you need anything else!
All the cricket equipment in our catalog—including the Cricket Bat, English Willow Bat, Kashmir Willow Bat, balls, gloves, pads, helmets, and accessories—are available in the same three colors:

- **Black**  
- **Blue**  
- **Red**  

Just let me know which color you prefer, and I’ll help you pick the right gear!
Here are all the items in our catalog priced **₹500 or less** (organized by sport):

**Cricket**  
- Crick

In [17]:
for message in history:

    if isinstance(message, HumanMessage):
        print("Customer:", message.content)

    elif isinstance(message, AIMessage):
        print("SportsMate AI:", message.content)

    print("-" * 60)

Customer: Hi, I'm Jahnavi, I need cricket equipment.
------------------------------------------------------------
SportsMate AI: Hi Jahnavi! 👋 I’d love to help you gear up for cricket. Are you looking for bats, gloves, pads, or something else? Let me know any specific items, preferred colors, or budget you have in mind, and I’ll suggest the best options from our catalog.
------------------------------------------------------------
Customer: How much is a cricket bat?
------------------------------------------------------------
SportsMate AI: The regular **Cricket Bat** is priced at **₹149**. It’s available in three colors:

- Black  
- Blue  
- Red  

If you’re interested in premium options, we also have the **English Willow Bat** (₹299) and the **Kashmir Willow Bat** (₹449), each in the same color choices. Let me know which one you’d like or if you need anything else!
------------------------------------------------------------
Customer: What colors are available?
--------------------

In [18]:
test_data = {
    "question": [
        "How much is a Cricket Bat?",
        "What colors are available in Football?",
        "How much is a Badminton Racket?",
        "Do you have Basketball?",
        "How much is a Yoga Mat?",
        "Which category does a Tennis Racket belong to?",
        "How much is a Skipping Rope?",
        "Do you have Dumbbells?"
    ],

    "must_include": [
        "149",
        "Black",
        "149",
        "299",
        "149",
        "Tennis",
        "149",
        "299"
    ]
}

df = pd.DataFrame(test_data)

df.to_excel(
    "/content/tests.xlsx",
    index=False
)

print("Excel test file created successfully!")

display(df)

Excel test file created successfully!


,question,must_include
0,How much is a Cricket Bat?,149
1,What colors are available in Football?,Black
2,How much is a Badminton Racket?,149
3,Do you have Basketball?,299
4,How much is a Yoga Mat?,149
5,Which category does a Tennis Racket belong to?,Tennis
6,How much is a Skipping Rope?,149
7,Do you have Dumbbells?,299


In [19]:
import re

def normalize(text):

    text = str(text).lower()

    text = text.replace("₹", "rs")
    text = text.replace(",", "")

    return re.sub(r"[^a-z0-9]", "", text)


def run_excel_tests(path="/content/tests.xlsx"):

    df = pd.read_excel(path, engine="openpyxl")

    passed = 0
    total = len(df)

    print("Excel file loaded successfully!\n")

    for _, row in df.iterrows():

        answer = chain.invoke({
            "catalog": catalog_text,
            "history": [],
            "question": row["question"]
        })

        expected = str(row["must_include"])

        ok = normalize(expected) in normalize(answer)

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"]
        )

        print("Answer:", answer)
        print("-" * 60)

    score = (passed / total * 100) if total else 0

    print("\n========== TEST RESULTS ==========")
    print(f"Passed: {passed}/{total}")
    print(f"Failed: {total-passed}/{total}")
    print(f"Success Rate: {score:.2f}%")


run_excel_tests()

Excel file loaded successfully!

PASS | How much is a Cricket Bat?
Answer: The **Cricket Bat** is priced at **₹149**. It’s available in **Black, Blue, and Red**, and is a standard bat suitable for all cricket activities. Let me know if you’d like to explore other cricket gear or need help choosing a color!
------------------------------------------------------------
PASS | What colors are available in Football?
Answer: The Football category offers three color options:

- **Black**
- **White**
- **Blue**

These colors are available for all Football products in our catalog. Let me know if you’d like details on any specific item!
------------------------------------------------------------
PASS | How much is a Badminton Racket?
Answer: The **Badminton Racket** is priced at **₹149**. It’s available in **Black, Red, and White** and is suitable for all badminton activities. Let me know if you’d like to see any accessories or other badminton gear!
---------------------------------------------

In [20]:
from langchain.agents import create_agent
from langchain.tools import tool

In [21]:
@tool
def search_sports_products(query: str) -> str:
    """
    Search the sports equipment catalog by product name,
    sports category, or budget.

    Examples:
    Cricket, Football, Badminton Racket,
    Yoga Mat, products under Rs 500.
    """

    query_lower = query.lower()

    # Budget-based search
    budget_match = re.search(
        r"(?:under|below|less than|within)\s*(?:rs\.?|₹)?\s*(\d+)",
        query_lower
    )

    results = {}

    if budget_match:

        budget = int(budget_match.group(1))

        results = {
            name: details
            for name, details in SPORTS_CATALOG.items()
            if details["price"] <= budget
        }

    else:

        # Product and category search
        for name, details in SPORTS_CATALOG.items():

            if (
                query_lower in name.lower()
                or query_lower in details["category"].lower()
            ):
                results[name] = details

    if not results:
        return "No matching products found in the catalog."

    return json.dumps(results, indent=2)

In [22]:
AGENT_SYSTEM = """
You are SportsMate AI, a friendly sports equipment shopping assistant.

RULES:

1. Use the search_sports_products tool for product information.
2. Always retrieve catalog details before answering questions
   about prices, colors, categories and availability.
3. Never invent product information.
4. Recommend products based on the customer's budget.
5. Respond in English or Telugu based on the user's language.
6. Remember customer preferences within the conversation.
7. If no product is found, clearly inform the customer.
8. Keep responses under 100 words.
9. Do not claim live inventory or stock information.

The catalog tool is your source of product facts.
"""

agent_order = create_agent(
    model=llm,
    tools=[search_sports_products],
    system_prompt=AGENT_SYSTEM
)

print("SportsMate AI Agent initialized successfully!")

SportsMate AI Agent initialized successfully!


In [23]:
result_order = agent_order.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you have badminton rackets? Tell me their prices and colors."
        }
    ]
})

print(result_order["messages"][-1].content)

Sure! Here are the badminton rackets we have:

| Product | Price (Rs) | Colors |
|---------|------------|--------|
| Badminton Racket | 149 | Black, Red, White |
| Professional Badminton Racket | 299 | Black, Red, White |

Let me know if you’d like more details or help choosing one!


In [24]:
result_order = agent_order.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Suggest sports equipment available under Rs 500."
        }
    ]
})

print(result_order["messages"][-1].content)

Here are some fun options under Rs 500:

| Product | Category | Price (Rs) | Color choices |
|---|---|---|---|
| Cricket Bat | Cricket | 149 | Black, Blue, Red |
| Football | Football | 199 | Black, White, Blue |
| Badminton Racket | Badminton | 149 | Black, Red, White |
| Tennis Racket | Tennis | 199 | Yellow, Black, Blue |
| Basketball | Basketball | 299 | Orange, Black |
| Yoga Mat | Fitness | 299 | Black, Pink, Blue |
| Skipping Rope | Fitness | 149 | Black, Pink, Blue |
| Dumbbells (pair) | Gym Equipment | 299 | Black, Grey, Red |
| Table Tennis Bat | Table Tennis | 99 | Red, Black |
| Volleyball | Volleyball | 199 | White, Blue, Yellow |

All are listed as “Available” in our catalog. Let me know if you’d like details on any item or help picking colors!


In [25]:
history.clear()

print(ask("I want to buy a football."))
print(ask("What is its price?"))
print(ask("Which colors are available?"))
print(ask("Suggest some badminton equipment."))

Sure! We have three footballs to choose from:

| Product | Price (₹) | Colors |
|---------|----------|--------|
| **Football** | 199 | Black, White, Blue |
| **Training Football** | 349 | Black, White, Blue |
| **Professional Football** | 499 | Black, White, Blue |

All are suitable for regular football activities. Do you have a preferred color or budget range? Let me know, and I’ll help you pick the best one!
The regular **Football** is priced at **₹199**. It’s available in Black, White, and Blue. Let me know if you’d like to choose a color or explore other options!
The Football comes in three colors:

- Black  
- White  
- Blue  

Let me know which color you prefer or if you’d like to see any other football‑related items!
Here are some popular badminton items you might like:

| Product | Price (₹) | Colors |
|---------|----------|--------|
| Badminton Racket | 149 | Black, Red, White |
| Professional Badminton Racket | 299 | Black, Red, White |
| Badminton Shuttlecock | 449 | Black, 

In [26]:
history.clear()

print(ask("Naaku cricket bat kavali."))

print(ask("Daani price entha?"))

print(ask("Inka em colors available unnayi?"))

print(ask("Rs 500 lopu sports items em unnayi?"))

Cricket bat kavālani cheppāru, mariyu mīku konni manchi vikalpalu unnāyi:

| Product | Price (₹) | Colors |
|---|---|---|
| **Cricket Bat** | 149 | Black, Blue, Red |
| **English Willow Bat** | 299 | Black, Blue, Red |
| **Kashmir Willow Bat** | 449 | Black, Blue, Red |

Mī budget, playing level, leda wood type (English vs Kashmir) yedaina vishēṣaṁ unte cheppandi, nēnu mīku saripoyina bat ni recommend chestānu!
Cricket Bat ki price **₹149**. Available colors: Black, Blue, Red. Any other bat type (English Willow ₹299, Kashmir Willow ₹449) kavālante cheppandi!
Prati cricket‑related item lo **Black, Blue, Red** ane 3 colors mātramē available. Kāvalante, i colors lo mee bat, gloves, pads, helmet, etc. anni select chesukovachu!
₹500 lopu mīku kavālani cricket items:

| Product | Price (₹) | Colors |
|---|---|---|
| Cricket Bat | 149 | Black, Blue, Red |
| English Willow Bat | 299 | Black, Blue, Red |
| Kashmir Willow Bat | 449 | Black, Blue, Red |

Ivi budget‑friendly options. Inka yedaina 

In [ ]:
print("=" * 60)
print("       WELCOME TO SPORTSMATE AI")
print("=" * 60)

print("Your Personal Sports Equipment Shopping Assistant")
print("Available in English and Telugu")
print("Type 'exit' to stop the chatbot.")
print("-" * 60)

history.clear()

while True:

    question = input("\nYou: ").strip()

    if question.lower() == "exit":

        print("\nSportsMate AI: Thank you for shopping with us!")
        print("Have a great day! 🏆")
        break

    if not question:
        continue

    try:

        answer = ask(question)

        print("\nSportsMate AI:", answer)

    except Exception as e:

        print("Error:", str(e))

       WELCOME TO SPORTSMATE AI
Your Personal Sports Equipment Shopping Assistant
Available in English and Telugu
Type 'exit' to stop the chatbot.
------------------------------------------------------------

You: hi i want a english willow criket bat

SportsMate AI: Sure! The **English Willow Bat** is a cricket bat priced at **₹299**. It’s available in **Black, Blue, and Red**. Let me know which color you prefer, or if you’d like any more details!
